# Driven: product and business

Plans, unit economics, a price sweep, the weekly leads report and launch readiness. Prices, rates and
flags all come from `product.json`, the single source the pipeline and the app share.

## 1. Plans

In [1]:
from dataclasses import asdict, replace

import numpy as np
import pandas as pd

from driven_ml.product import UsageAssumptions, load_product
from driven_ml.settings import find_repo_root

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)
repo_root = find_repo_root()
product = load_product(repo_root)
plans = pd.DataFrame([asdict(plan) for plan in product.plans])
plans["features"] = plans["features"].map(", ".join)
plans

,plan_id,name,price_usd_month,parent_sms,summary,features
0,pilot,Pilot,0.00,True,Free for the first DFW families while we tune the model with them.,"In-car spoken alerts, Parent text alerts, Drive history"
1,family,Family,9.99,True,"One teen driver, up to two parent or emergency contacts.","In-car spoken alerts, Parent text alerts, Drive history, Weekly drive summary"
2,free,In-car only,0.00,False,Spoken alerts in the car. No parent texts or history.,In-car spoken alerts
3,partner,Partners,NaN,True,"Dashcam makers, insurers and driver-education programs: licensing and pilots.","Model licensing, Aggregate, opt-in safety reports, Co-branded app"


## 2. Unit economics of the Family plan

Each false alert texts every confirmed contact, so false alerts cost money as well as trust. The
staging gate checks the margin at the model's measured test false-alert rate; the break-even rate is
the most false alerts per drive hour the price can absorb at the target margin.

In [2]:
from driven_ml.economics import break_even_false_alert_rate, unit_economics
from driven_ml.spec import FALSE_ALERT_BUDGET_PER_HOUR

rates = [0.0, 0.2, 1.0, 2.0, 4.0]
economics = pd.DataFrame([asdict(unit_economics(product, rate)) for rate in rates])
display(economics.round(3))
print(
    f"Break-even: {break_even_false_alert_rate(product):.2f} false alerts per drive hour "
    f"at a {product.min_contribution_margin:.0%} margin target"
)
print("Still team estimates, to replace with measured pilot values:", ", ".join(UsageAssumptions.names()))
print(product.raw["assumptions"]["note"])

,plan_id,price_usd_month,false_alerts_per_hour,texts_per_month,sms_usd_month,infrastructure_usd_month,payments_usd_month,cost_usd_month,contribution_margin
0,family,9.99,0.0,4.0,0.053,0.523,0.59,1.166,0.883
1,family,9.99,0.2,12.0,0.160,0.523,0.59,1.272,0.873
2,family,9.99,1.0,44.0,0.585,0.523,0.59,1.698,0.830
3,family,9.99,2.0,84.0,1.117,0.523,0.59,2.230,0.777
4,family,9.99,4.0,164.0,2.181,0.523,0.59,3.294,0.670


Break-even: 5.32 false alerts per drive hour at a 60% margin target
Still team estimates, to replace with measured pilot values: drive_hours_per_month, true_drowsy_alerts_per_drive_hour, confirmed_contacts_per_family, segments_per_sms, paying_families
Team estimates for planning. Replace each with the measured pilot value; the economics report prints which ones are still estimates.


## 3. Price sweep

The contribution margin of the Family plan at the 0.2 false alerts per hour budget, for four prices.

In [3]:
def with_family_price(price_usd_month: float):
    plans = tuple(
        replace(plan, price_usd_month=price_usd_month) if plan.plan_id == "family" else plan for plan in product.plans
    )
    return replace(product, plans=plans)


sweep = pd.DataFrame(
    [
        {
            "price_usd_month": price,
            "margin_at_budget": unit_economics(
                with_family_price(price), FALSE_ALERT_BUDGET_PER_HOUR
            ).contribution_margin,
            "break_even_false_alerts_per_hour": break_even_false_alert_rate(with_family_price(price)),
        }
        for price in (4.99, 7.99, 9.99, 12.99)
    ]
)
sweep.round(3)

,price_usd_month,margin_at_budget,break_even_false_alerts_per_hour
0,4.99,0.774,1.833
1,7.99,0.848,3.925
2,9.99,0.873,5.320
3,12.99,0.895,7.412


## 4. Leads report

The landing page's lead export, summarized for the weekly go-to-market review. This example export
is generated; point `summarize_leads` at the real CSV from the backend instead.

In [4]:
from driven_ml.leads import summarize_leads

rng = np.random.default_rng(4)
count = 60
example = pd.DataFrame(
    {
        "created_at": (
            pd.Timestamp("2026-10-05", tz="UTC") + pd.to_timedelta(rng.uniform(0, 28 * 24, count), unit="h")
        ).floor("s"),
        "email": [f"lead{i}@example.com" for i in range(count)],
        "role": rng.choice(
            ["parent", "insurer", "dashcam_maker", "driver_education"], count, p=[0.82, 0.08, 0.05, 0.05]
        ),
        "state": rng.choice(["TX", "OK", "LA", "NM", "CA"], count, p=[0.7, 0.1, 0.08, 0.07, 0.05]),
        "utm_source": rng.choice(["instagram", "school_flyer", "linkedin", None], count, p=[0.4, 0.3, 0.15, 0.15]),
        "teen_drivers": rng.integers(0, 3, count),
        "sms_marketing_consent": rng.random(count) < 0.45,
    }
)
leads = summarize_leads(example)
print(f"{leads.total_leads} leads, {leads.teen_drivers} teen drivers")
print(f"{leads.sms_opt_in_rate:.0%} opted in to marketing texts")
display(leads.by_week)
display(leads.by_role)
display(leads.by_source)
display(leads.by_state)
leads.partner_leads

60 leads, 56 teen drivers
33% opted in to marketing texts


,week_start,leads
0,2026-10-05,10
1,2026-10-12,17
2,2026-10-19,14
3,2026-10-26,19


,role,leads
0,parent,52
1,dashcam_maker,4
2,insurer,3
3,driver_education,1


,utm_source,leads,parents,partners
0,instagram,22,17,5
1,linkedin,14,12,2
2,direct,13,12,1
3,school_flyer,11,11,0


,state,leads
0,TX,35
1,OK,10
2,LA,5
3,NM,5
4,CA,5


,created_at,email,role,state,utm_source
0,2026-11-01 08:02:08+00:00,lead2@example.com,dashcam_maker,LA,direct
1,2026-10-30 14:55:06+00:00,lead52@example.com,dashcam_maker,CA,instagram
2,2026-10-29 09:44:20+00:00,lead8@example.com,insurer,TX,instagram
3,2026-10-27 23:13:11+00:00,lead54@example.com,insurer,TX,instagram
4,2026-10-20 05:31:43+00:00,lead9@example.com,dashcam_maker,TX,linkedin
5,2026-10-19 07:36:43+00:00,lead1@example.com,driver_education,TX,instagram
6,2026-10-18 07:53:53+00:00,lead38@example.com,dashcam_maker,TX,instagram
7,2026-10-17 00:35:49+00:00,lead59@example.com,insurer,TX,linkedin


## 5. Launch readiness

Everything that must be true before real families use Driven. In this ML-only repository most rows
fail, which is correct: there is no app and no released real-data model yet. Each failing row says
what it needs.

In [5]:
from driven_ml.launch import readiness_checks

checks = pd.DataFrame([asdict(check) for check in readiness_checks(repo_root)])
display(checks[["name", "passed", "detail"]])
checks.loc[~checks["passed"], ["name", "how_to_fix"]]

,name,passed,detail
0,product flags pass compliance,True,L1-L7 and G1 satisfied
1,"landing page, app and backend scan clean",False,0 files scanned
2,backend copy of product.json in sync,False,supabase/functions/_shared/product.json not found
3,privacy contact email set,False,contact_email is empty
4,Stripe Payment Link set,False,billing.stripe_payment_link is empty
5,released model in the app,False,/home/claude/v2/apps/web/public/models/driven-current.json not found
6,"released model uses real, commercial-track data",False,no released model
7,released model matches the feature spec version,False,no released model
8,Face Landmarker model self-hosted,False,apps/web/public/mediapipe/face_landmarker.task not found
9,app parity fixtures present,False,"missing: model.json, feature_spec.json, frame_fixtures.json, window_fixtures.json, replay_fixture.json, parity_fixtures.json"


,name,how_to_fix
1,"landing page, app and backend scan clean","build the landing page and app in apps/web, then remove every flagged claim or API"
2,backend copy of product.json in sync,run `make sync` in the app repository
3,privacy contact email set,"set contact_email in product.json (TDPSA, Tex. Bus. & Com. Code 541.102)"
4,Stripe Payment Link set,create a Payment Link for the Family plan in Stripe and set billing.stripe_payment_link
5,released model in the app,promote a staged model: python -m driven_ml --env production --track commercial --approved-by <name> --write-app
6,"released model uses real, commercial-track data",train on consented team recordings or commercially licensed data with --track commercial
7,released model matches the feature spec version,retrain and promote with the current feature spec
8,Face Landmarker model self-hosted,download face_landmarker.task into apps/web/public/mediapipe/ so no video frame leaves the phone
9,app parity fixtures present,run a staging or dev pipeline with --write-app and make the app's parity tests pass
